In [6]:
# ============================================================
# DATA LOADING v5 — BTC/ETH + Macro Signals
#
# Signals:
#   + SPY daily log return   (risk-on/off regime)
#   + DXY daily log return   (dollar strength, inverse BTC)
#   + VIX level + 1d change  (market fear/volatility)
#   + Fear & Greed Index     (crypto sentiment, alternative.me)
#
# Rules:
#   - Base columns untouched: timestamp, price, volume, coin
#   - NO preprocessing, NaN handling, or forward-filling
#   - Raw LEFT JOIN only — NaNs handled in EDA/preprocessing
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import requests
import os

COINS         = ["BTC-USD", "ETH-USD"]
START_DATE    = "2017-01-01"
END_DATE      = "2026-02-12"
RAW_DATA_PATH = "../data/raw"
os.makedirs(RAW_DATA_PATH, exist_ok=True)


# ─────────────────────────────────────────────────────────────
# HELPER — download single yfinance ticker
# ─────────────────────────────────────────────────────────────
def download_yf(ticker, start, end, col_rename):
    df = yf.download(ticker, start=start, end=end,
                     interval="1d", auto_adjust=True, progress=False)
    if df.empty:
        print(f"  ⚠️  {ticker} returned empty!")
        return None
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()[["Date", "Close"]].rename(
        columns={"Date": "timestamp", "Close": col_rename}
    )
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    print(f"  ✅ {ticker}: {len(df)} rows | "
          f"{df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
    return df


# ═══════════════════════════════════════════════════════════════
# BLOCK 1 — CRYPTO PRICES
# ═══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📥 BLOCK 1 — CRYPTO PRICES")
print("="*60)

all_crypto = []
for ticker in COINS:
    print(f"\n  Downloading {ticker}...")
    df = yf.download(ticker, start=START_DATE, end=END_DATE,
                     interval="1d", auto_adjust=True, progress=False)
    if df.empty:
        print(f"  ⚠️  {ticker} empty — skipping")
        continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    df = df.rename(columns={"Date":"timestamp", "Close":"price", "Volume":"volume"})
    df = df[["timestamp", "price", "volume"]]
    df["coin"] = ticker.split("-")[0].lower()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    print(f"  ✅ {ticker}: {len(df)} rows")
    all_crypto.append(df)

crypto_df = pd.concat(all_crypto, ignore_index=True)
crypto_df = crypto_df.sort_values(["coin", "timestamp"]).reset_index(drop=True)
print(f"\n  Combined crypto shape: {crypto_df.shape}")


# ═══════════════════════════════════════════════════════════════
# BLOCK 2 — MACRO SIGNALS via yfinance
# ═══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📥 BLOCK 2 — MACRO SIGNALS (yfinance)")
print("="*60)

spy_df = download_yf("SPY",      START_DATE, END_DATE, "spy_close")
dxy_df = download_yf("DX-Y.NYB", START_DATE, END_DATE, "dxy_close")
vix_df = download_yf("^VIX",     START_DATE, END_DATE, "vix")


# ═══════════════════════════════════════════════════════════════
# BLOCK 3 — FEAR & GREED INDEX (alternative.me)
# ═══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📥 BLOCK 3 — FEAR & GREED INDEX (alternative.me)")
print("="*60)

def fetch_fear_greed(limit=3000):
    url = f"https://api.alternative.me/fng/?limit={limit}&format=json"
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        data = resp.json()["data"]
        df = pd.DataFrame(data)[["timestamp", "value"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"].astype(int), unit="s")
        df["timestamp"] = df["timestamp"].dt.normalize()
        df["fear_greed"] = df["value"].astype(float)
        df = df[["timestamp", "fear_greed"]].sort_values("timestamp").reset_index(drop=True)
        print(f"  ✅ Fear & Greed: {len(df)} rows | "
              f"{df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
        return df
    except Exception as e:
        print(f"  ❌ Fear & Greed fetch failed: {e}")
        return None

fg_df = fetch_fear_greed(limit=3000)


# ═══════════════════════════════════════════════════════════════
# BLOCK 4 — MACRO RETURNS
# ═══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📥 BLOCK 4 — COMPUTING MACRO RETURNS")
print("="*60)

def add_log_return(df, close_col, return_col):
    df = df.sort_values("timestamp").copy()
    df[return_col] = np.log(df[close_col] / df[close_col].shift(1))
    df = df.drop(columns=[close_col])
    return df

if spy_df is not None:
    spy_df = add_log_return(spy_df, "spy_close", "spy_return")
    print(f"  SPY return — mean: {spy_df['spy_return'].mean():.5f} | "
          f"std: {spy_df['spy_return'].std():.5f}")

if dxy_df is not None:
    dxy_df = add_log_return(dxy_df, "dxy_close", "dxy_return")
    print(f"  DXY return — mean: {dxy_df['dxy_return'].mean():.5f} | "
          f"std: {dxy_df['dxy_return'].std():.5f}")

if vix_df is not None:
    vix_df["vix_change"] = vix_df["vix"].diff()
    print(f"  VIX — mean: {vix_df['vix'].mean():.2f} | "
          f"std: {vix_df['vix'].std():.2f}")


# ═══════════════════════════════════════════════════════════════
# BLOCK 5 — MERGE  (raw LEFT JOIN, no filling)
# ═══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📥 BLOCK 5 — MERGING ALL SIGNALS")
print("="*60)

df = crypto_df.copy()

for name, ext_df in [
    ("SPY",        spy_df),
    ("DXY",        dxy_df),
    ("VIX",        vix_df),
    ("Fear&Greed", fg_df),
]:
    if ext_df is None:
        print(f"  ⚠️  {name} skipped (download failed)")
        continue
    before = df.shape[1]
    df = df.merge(ext_df, on="timestamp", how="left")
    print(f"  ✅ {name} merged — +{df.shape[1] - before} col(s)")


# ═══════════════════════════════════════════════════════════════
# BLOCK 6 — SUMMARY & SAVE
# ═══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("✅ FINAL DATASET SUMMARY")
print("="*60)
print(f"  Shape      : {df.shape}")
print(f"  Coins      : {df['coin'].value_counts().to_dict()}")
print(f"  Date range : {df['timestamp'].min().date()} → "
      f"{df['timestamp'].max().date()}")
print(f"\n  Columns & NaN counts:")
nan_counts = df.isna().sum()
for col in df.columns:
    pct = nan_counts[col] / len(df) * 100
    flag = f"  ← {pct:.1f}% NaN" if pct > 0 else ""
    print(f"    {col:<30} {flag}")

print(f"\n  Sample (last 3 rows per coin):")
print(df.groupby("coin").tail(3).to_string(index=False))

file_path = os.path.join(RAW_DATA_PATH, "crypto_raw.csv")
df.to_csv(file_path, index=False)
print(f"\n✅ Saved: {file_path}")


📥 BLOCK 1 — CRYPTO PRICES

  ✅ BTC-USD: 3329 rows

  ✅ ETH-USD: 3017 rows

  Combined crypto shape: (6346, 4)

📥 BLOCK 2 — MACRO SIGNALS (yfinance)
  ✅ SPY: 2290 rows | 2017-01-03 → 2026-02-11
  ✅ DX-Y.NYB: 2292 rows | 2017-01-03 → 2026-02-11
  ✅ ^VIX: 2290 rows | 2017-01-03 → 2026-02-11

📥 BLOCK 3 — FEAR & GREED INDEX (alternative.me)
  ✅ Fear & Greed: 2931 rows | 2018-02-01 → 2026-02-13

📥 BLOCK 4 — COMPUTING MACRO RETURNS
  SPY return — mean: 0.00055 | std: 0.01163
  DXY return — mean: -0.00003 | std: 0.00416
  VIX — mean: 18.77 | std: 7.52

📥 BLOCK 5 — MERGING ALL SIGNALS
  ✅ SPY merged — +1 col(s)
  ✅ DXY merged — +1 col(s)
  ✅ VIX merged — +2 col(s)
  ✅ Fear&Greed merged — +1 col(s)

✅ FINAL DATASET SUMMARY
  Shape      : (6346, 9)
  Coins      : {'btc': 3329, 'eth': 3017}
  Date range : 2017-01-01 → 2026-02-11

  Columns & NaN counts:
    timestamp                      
    price                          
    volume                         
    coin                           
 